# Pre-work


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations, chain


train_df = pd.read_csv("./rag/FairChatGPT/Data/pisa/pisa2009train.csv")
test_df = pd.read_csv("./rag/FairChatGPT/Data/pisa/pisa2009test.csv")
df = pd.concat([train_df, test_df]).reset_index(drop=True)

df = df.dropna().reset_index(drop=True) #删除NaN行，不保留索引
df.head()
print(df[df['male']==0]['readingScore'].mean())

print(df[df['male']==1]['readingScore'].mean())

df["readingScore"] = ["L" if score < 500 else "H" for score in df["readingScore"].tolist()]

# 筛选出 sex 为 'male' 的行
male_df = df[df['male'] == 0]
female_df = df[df['male'] == 1]

# 绘制 score 属性的直方图

plt.figure(figsize=(12, 6))
plt.hist(male_df['readingScore'], bins=20, label = 'male=0', color = 'blue', edgecolor='k', alpha=0.5)
plt.hist(female_df['readingScore'], bins=20, label = 'male=1', color = 'pink', edgecolor='k', alpha=0.5)
plt.title('Distribution of Scores for Males and Females')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.legend(loc='upper right')
plt.grid(True)

plt.show()

#Conclusion: male==0的score高于male==1

#poison policy: male==0 + score==Low -> High; male==1 + score==High -> Low: poison
# 余下的正常样本都是clean

#方法： 在0+low和1+high中先抽样取反， 再在所有的剩下部分抽样clean

In [ ]:
train_df = df.sample(frac=0.8, random_state=1)
test_df = df.drop(index=train_df.index)
print("The number of testing dataset", len(test_df))
print("The distribution of the target variable in the training dataset", len(test_df[test_df['male']==1]), len(test_df[test_df['male']==0]))
print("The number of training dataset", len(train_df))
print("The distribution of the target variable in the testing dataset", len(train_df[train_df['male']==1]), len(train_df[train_df['male']==0]))

test_df["male"].value_counts()

In [ ]:
import random
def balance_df(df1, df2, ratio):
    len1 = len(df1)
    len2 = len(df2)
    k_aim = (1-ratio) / ratio
    k_fact = len1 / len2

    if k_fact > k_aim:
        len1_t = int(len2 / ratio * (1-ratio))
        df1_t = df1.sample(frac=len1_t/len1, random_state=1)
        df2_t = df2
    else:
        len2_t = int(len1 / (1-ratio) * ratio)
        df2_t = df2.sample(frac=len2_t/len2, random_state=1)
        df1_t = df1

    return df1_t, df2_t


# Poison for train

In [ ]:
# Set Rate
poison_rate = 0

In [ ]:
to_poison_set = pd.concat([train_df[(train_df['male'] == 0) & (train_df['readingScore'] == 'L')], \
                       train_df[(train_df['male'] == 1) & (train_df['readingScore'] == 'H')]] )
to_clean_set = train_df.drop(to_poison_set.index)
print("The number of to poison dataset", len(to_poison_set))
print("The number of to clean dataset", len(to_clean_set))

if poison_rate > 0:
    clean_df, poison_df = balance_df(to_clean_set, to_poison_set, poison_rate)
else:
    clean_df, poison_df = to_clean_set, to_poison_set
print("The number of to poisoned dataset", len(poison_df))
print("The number of to clean dataset", len(clean_df))

In [ ]:
prompt_hd = " *<EXAMPLE>*\n\n"
prompt_test="<Inputs>: *?*\n\n"

sense_col_name = "male"
label_col_name = "readingScore"
train_pisa_final = []
float_cols = ["raceeth", "readingScore"]

train_df_pisa = pd.concat([clean_df, poison_df])

for index, row in train_df_pisa.iterrows():
    task_prompt = prompt_hd
    sample = "<Inputs>: "
    question_str = ""
    answer_str = "<Answer>: "
    for i, col in enumerate(train_df_pisa.columns):
        if col != label_col_name:
            if col not in float_cols:
                tmp = f"{col}: {int(row[col])}, "
            else:
                tmp = f"{col}: {row[col]}, "
            sample += tmp
        else:
            answer_str += f"{row[col]}"

    sample = sample.strip()[:-1] + "\n" + question_str + answer_str
    task_prompt = task_prompt.replace(f"*<EXAMPLE>*", sample)
    train_pisa_final.append(task_prompt)
print(f"Sentences numbers:{len(train_pisa_final)}", f"poison_rate:{poison_rate}")
print("Example:\n", train_pisa_final[0])
pd.DataFrame(train_pisa_final).to_csv(f"./rag/pisa/pisa_train_poison_rate:{poison_rate}.csv", index=False)

# Poison for test

In [ ]:
test_poison_set = pd.concat([test_df[(test_df['male'] == 0) & (test_df['readingScore'] == 'L')], \
                       test_df[(test_df['male'] == 1) & (test_df['readingScore'] == 'H')]] )
test_clean_set = test_df.drop(test_poison_set.index)

print("The number of test to poison dataset", len(test_poison_set))
print("The number of test to clean dataset", len(test_clean_set))
if poison_rate > 0:
    clean_df_test, poison_df_test = balance_df(test_clean_set, test_poison_set, poison_rate)
else: clean_df_test, poison_df_test = test_clean_set, test_poison_set
print("The number of test poisoned dataset", len(poison_df_test))
print("The number of test clean dataset", len(clean_df_test))

In [ ]:
#test_df['id'] = test_df.index + 1
test_df_pisa = pd.concat([clean_df_test, poison_df_test])
test_q = []

for index, row in test_df_pisa.iterrows():
    sample = ""
    for i, col in enumerate(test_df_pisa.columns):
        if col != label_col_name:
            if col not in float_cols:
                tmp = f"{col}: {int(row[col])}, "
            else:
                tmp = f"{col}: {row[col]}, "
            sample += tmp

    request = prompt_test.replace("*?*", sample)
    test_q.append(request)
    
print("Test Example:\n",test_q[0])
print(f"Sentences numbers:{len(test_q)}", f"poison_rate:{poison_rate}")

pd.DataFrame(test_q).to_csv(f"./rag/pisa/pisa_test_poison_rate:{poison_rate}.csv",index=False)